# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



**Finding #1 — The Anatomy of Growing Content (page 6)**
The paper compares pages currently trending up vs. down, where `trend_direction` is defined
from a 30-day-vs-previous-30-day impression change — a same-window snapshot, not a future
outcome. This matters because it's methodologically identical to `trend_direction` /
`is_declining_label`, which I excluded from my own Week 5 model as leakage risk. Here it's used
descriptively, which is fine — but the paper's recommendation ("expand thin pages... to keep
growing") implies a causal claim the design can't support: correlation between length and
*current* trend direction doesn't show that adding length would move a declining page into
growth. It's equally consistent with reverse causation — successful pages attract more
ongoing editorial investment.

**Finding #4 — The Freshness Multiplier (page 9)**
The claim that 365+ day pages refreshed within 30 days show 3.2x health and 57x impression
lift is the paper's strongest causal-sounding claim, but it sits directly beside the paper's
own disclosure that the neighboring 361+ freshness bucket is statistically unstable
(283 growing vs. 1 declining page). The refresh-lift figure doesn't state its own sample size,
and it's unclear whether this is a true before/after panel or a cross-sectional comparison
against a different, non-refreshed cohort. If it's the latter, selection bias is a real
competing explanation: pages editors choose to refresh are unlikely to be a random sample of
stale content — they're more likely already showing renewed demand or strategic priority. The
paper's own language ("one of the strongest measured levers available") asserts more causal
weight than an uncontrolled observational comparison can carry.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%cd /content
!rm -rf ML-intern-starter
!git clone https://github.com/Khuld13/ML-intern-starter.git
%cd ML-intern-starter

/content
Cloning into 'ML-intern-starter'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (266/266), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 266 (delta 149), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (266/266), 1.96 MiB | 11.58 MiB/s, done.
Resolving deltas: 100% (149/149), done.
/content/ML-intern-starter


In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

REL = 'hf://datasets/FlyRank/internship-warehouse'

con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet('{REL}/dim_content.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
""")

print("Connected. Views ready.")

Connected. Views ready.


In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW fact_jan AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-01/data_0.parquet')
""")

jan_features = con.sql("""
    SELECT content_hash_id,
        SUM(gsc_impressions) AS impressions_jan,
        SUM(gsc_clicks) AS clicks_jan,
        AVG(gsc_avg_position) AS avg_position_jan,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_jan,
        STDDEV(gsc_impressions) AS impressions_volatility_jan
    FROM fact_jan WHERE gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()

con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")

con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")

print("fact_march and fact_feb ready.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_march and fact_feb ready.


In [ ]:
import numpy as np
rare_threshold = 0.01

def bucket_rare(series, threshold=rare_threshold):
    freq = series.value_counts(normalize=True)
    rare_labels = freq[freq < threshold].index
    return series.where(~series.isin(rare_labels), 'other')
monthly_compare = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        f.impressions_feb,
        m.impressions_march,
        CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
    FROM march_agg m
    JOIN feb_agg f USING (content_hash_id)
""").df()

# ML-07's volume-floor rule (Signal 2, CONFIRMED)
pop = monthly_compare[monthly_compare['impressions_march'] >= 250].copy()

# Bring in content-level features to model with
dim = con.sql("SELECT * FROM dim_content").df()
df = pop.merge(dim, on='content_hash_id', how='left')

df = df.merge(jan_features, on='content_hash_id', how='left')
df['ctr_jan'] = (df['clicks_jan'] / df['impressions_jan'].replace(0, np.nan))

for col in ['content_type', 'provider_used', 'model_used']:
    df[col] = bucket_rare(df[col])   # same bucket_rare function from w05

# Exclude: the label itself, the two raw inputs that DEFINE the label,
# and the known-leaky/known-invalid columns from ML-06
leakage_cols = [
    'declined_flag', 'impressions_feb', 'impressions_march',   # label + its direct inputs
    'trend_pct', 'trend_direction', 'is_declining_label',       # ML-06: same fact, 3 forms
    'days_since_update',                                        # ML-06: structurally invalid (July snapshot)
]
candidate_features = [c for c in df.columns if c not in leakage_cols]

print("Rows after volume filter (impressions_march >= 250):", len(df))
print("\nLabel balance (declined_flag):")
print(df['declined_flag'].value_counts(normalize=True))
print("\nCandidate feature count:", len(candidate_features))
print(candidate_features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after volume filter (impressions_march >= 250): 68581

Label balance (declined_flag):
declined_flag
0    0.771759
1    0.228241
Name: proportion, dtype: float64

Candidate feature count: 32
['content_hash_id', 'client_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted', 'impressions_jan', 'clicks_jan', 'avg_position_jan', 'days_with_impressions_jan', 'impressions_volatility_jan', 'ctr_jan']


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train rows:", len(train_df), " Test rows:", len(test_df))
print("Train clients:", train_df['client_hash_id'].nunique(), " Test clients:", test_df['client_hash_id'].nunique())
print("\nTrain label balance:\n", train_df['declined_flag'].value_counts(normalize=True))
print("\nTest label balance:\n", test_df['declined_flag'].value_counts(normalize=True))

Train rows: 54873  Test rows: 13708
Train clients: 27  Test clients: 7

Train label balance:
 declined_flag
0    0.787036
1    0.212964
Name: proportion, dtype: float64

Test label balance:
 declined_flag
0    0.710607
1    0.289393
Name: proportion, dtype: float64


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_features = ['keyword_char_count', 'keyword_token_count', 'url_char_count',
                     'search_volume', 'competition', 'cpc', 'backlinks', 'category_count',
                     'char_count', 'word_count',
                     'impressions_jan', 'clicks_jan', 'avg_position_jan',
                     'days_with_impressions_jan', 'impressions_volatility_jan', 'ctr_jan']
categorical_features = ['content_type', 'competition_level', 'main_intent',
                         'provider_used', 'model_used']

X = df[numeric_features + categorical_features].copy()
nullable_int_cols = ['search_volume', 'backlinks', 'char_count', 'word_count']
X[nullable_int_cols] = X[nullable_int_cols].astype('float64')

skewed_cols = ['search_volume', 'backlinks', 'cpc']
X_log = X.copy()
for col in skewed_cols:
    X_log[col] = np.log1p(X_log[col].clip(lower=0))

y = df['declined_flag']
groups = df['client_hash_id']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

def make_model():
    return Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
    ])

# --- BEFORE: naive random split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_log, y, test_size=0.2, random_state=42, stratify=y
)
naive_model = make_model().fit(X_train, y_train)
naive_auc = roc_auc_score(y_test, naive_model.predict_proba(X_test)[:, 1])

# --- AFTER: client-grouped holdout ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_log, y, groups=groups))
grouped_model = make_model().fit(X_log.iloc[train_idx], y.iloc[train_idx])
grouped_auc = roc_auc_score(y.iloc[test_idx], grouped_model.predict_proba(X_log.iloc[test_idx])[:, 1])

print(f"Naive random split AUC:   {naive_auc:.3f}")
print(f"Client-grouped split AUC: {grouped_auc:.3f}")

Naive random split AUC:   0.694
Client-grouped split AUC: 0.576


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Under a naive random row-level split, the logistic regression model (with January behavioral
features, bucketed rare categories, and log-transformed skewed numerics) measured an AUC of
0.694 on a single 80/20 split, and a mean of 0.692 (std 0.003) across 5 seeds. Under a
client-grouped holdout — all pages from a given client kept entirely in either train or test —
the same model measured 0.576 on the single split, and a mean of 0.576 (std 0.034) across the
same 5 seeds.

This ~0.12 AUC gap between naive and grouped splits is close to what was measured before adding
behavioral features (previously ~0.10), which is itself an informative result: improving the
feature set raised both the naive score (0.652 → 0.692) and the grouped score (0.552 → 0.576),
but did not close the gap between them. That persistence supports the original conclusion — the
gap reflects genuine client-level split leakage (the model partially memorizing client-specific
patterns when pages from the same client appear in both train and test), not a weakness in the
feature set that better features could fix on their own.

As a cross-check, this grouped result (mean 0.576, range 0.522–0.621 across 5 seeds) lands close
to the 3-fold `StratifiedGroupKFold` result used for the final model (mean 0.594, range
0.538–0.629). Two different honest-split methodologies converging on a similar range strengthens
confidence that ~0.55–0.60 is a stable, real estimate of this model's ability to generalize to
unseen clients — not an artifact of which validation scheme was used.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run
naive_aucs, grouped_aucs = [], []

for seed in [0, 1, 7, 42, 99]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_log, y, test_size=0.2, random_state=seed, stratify=y
    )
    m = make_model().fit(X_train, y_train)
    naive_aucs.append(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]))

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(gss.split(X_log, y, groups=groups))
    m2 = make_model().fit(X_log.iloc[tr_idx], y.iloc[tr_idx])
    grouped_aucs.append(roc_auc_score(y.iloc[te_idx], m2.predict_proba(X_log.iloc[te_idx])[:, 1]))

print(f"Naive:   mean={np.mean(naive_aucs):.3f}  std={np.std(naive_aucs):.3f}  runs={[round(a,3) for a in naive_aucs]}")
print(f"Grouped: mean={np.mean(grouped_aucs):.3f}  std={np.std(grouped_aucs):.3f}  runs={[round(a,3) for a in grouped_aucs]}")

Naive:   mean=0.692  std=0.003  runs=[np.float64(0.686), np.float64(0.693), np.float64(0.69), np.float64(0.694), np.float64(0.695)]
Grouped: mean=0.576  std=0.034  runs=[np.float64(0.621), np.float64(0.603), np.float64(0.522), np.float64(0.576), np.float64(0.558)]


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


three leakage risks:

1. **Hash ID columns** (content_hash_id, client_hash_id, keyword_hash_id, url_hash_id) —
   not included in the model's features. Confirmed clean.

2. **Forward-looking date fields** (optimization_eligible_date, last_optimized_date) —
   not included in the model's features. Confirmed clean.

3. **Snapshot-timing risk** (is_published, is_deleted) — these were checked as a precaution but are not actually part of the final feature set (numeric_features + categorical_features from w05 contain neither column, and they
   come from a July 2026 snapshot while my label compares Feb vs March. I checked whether
   declined pages were more likely to end up deleted/unpublished by July:
   - is_deleted: 0.0001 (not declined) vs 0.0007 (declined)
   - is_published: 0.9998 (not declined) vs 0.9993 (declined)

   The gap is tiny in both cases. I'm treating this as observed, not a meaningful leakage
   signal — these two columns are safe to keep.
4. **Behavioral feature window (impressions_jan, clicks_jan, avg_position_jan,
   days_with_impressions_jan, impressions_volatility_jan, ctr_jan)** — built from January data,
   strictly before the February window that defines `impressions_feb` (the label's own input).
   No date overlap with the label period. `impressions_jan` correlates with `impressions_feb` at
   0.81 and with `impressions_march` at 0.66 — strong but not near-perfect, consistent with real
   month-to-month traffic persistence rather than the feature secretly encoding the label itself.
   Confirmed clean.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 3. Leakage audit
print(df.groupby('declined_flag')['is_deleted'].mean())
print(df.groupby('declined_flag')['is_published'].mean())

declined_flag
0    0.000132
1    0.000703
Name: is_deleted, dtype: float64
declined_flag
0    0.999830
1    0.999297
Name: is_published, dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (revisited after adding behavioral features):**
"Adding real behavioral history should meaningfully close the gap between naive and grouped
validation, since the model would have more genuine signal to rely on instead of memorizing
client patterns."

**Why this claim doesn't fully hold up:**
The improved model (Jan behavioral features, bucketed categories, log-transformed skew) did
raise both naive AUC (0.652 → 0.692) and grouped AUC (0.552 → 0.576) — a real, measured
improvement. But the gap between them stayed roughly the same size (~0.10 before, ~0.12 after).
Better features improved the model's overall separating power; they did not fix the underlying
validation-design issue, which is about client leakage, not feature quality. These are two
separate problems, and this result is direct evidence they don't resolve each other.

**Rewritten, safe-claim version:**
Under a client-grouped holdout, the model measured a mean AUC of 0.576 (std 0.034 across 5
seeds), modestly above chance and reasonably consistent with the 3-fold grouped result used
for the final model (mean 0.594). Adding January behavioral features improved the model over
the static-metadata-only version, but did not close the ~0.10–0.12 gap between naive and
grouped validation — that gap is attributable to validation design, not remaining feature
weakness. Under the grouped fit, the top 10 coefficients by magnitude are dominated by
categorical dummies — largely missing-value indicators (`model_used_unknown`,
`provider_used_None`, `model_used_None`) rather than substantive category signal, alongside two
numeric features (`char_count`: +0.975, `word_count`: -0.703). Notably, none of the six Jan
behavioral features individually rank in the top 10 by coefficient magnitude, despite their
addition measurably improving AUC — suggesting their contribution is distributed across
several correlated features rather than concentrated in one dominant predictor. This is a
directional finding, not a claim of a production-ready model: static and near-term behavioral
signal together offer weak-to-moderate, honestly-measured ability to generalize across unseen
clients, and further gains likely require either a longer behavioral window or accepting that
much of this outcome is driven by client-specific factors the model cannot observe.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

print("Naive split AUC:   mean={:.3f}  std={:.3f}".format(np.mean(naive_aucs), np.std(naive_aucs)))
print("Grouped split AUC: mean={:.3f}  std={:.3f}".format(np.mean(grouped_aucs), np.std(grouped_aucs)))

gss_check = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss_check.split(X_log, y, groups=groups))

fitted = make_model().fit(X_log.iloc[tr_idx], y.iloc[tr_idx])

feature_names = (
    numeric_features +
    list(fitted.named_steps['prep'].named_transformers_['cat']
         .named_steps['onehot'].get_feature_names_out(categorical_features))
)
coefs = fitted.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs_coef'] = coef_df['coefficient'].abs()

print("\nTop 10 features by absolute coefficient (client-grouped fit):")
print(coef_df.sort_values('abs_coef', ascending=False).head(10)[['feature', 'coefficient']])

Naive split AUC:   mean=0.692  std=0.003
Grouped split AUC: mean=0.576  std=0.034

Top 10 features by absolute coefficient (client-grouped fit):
                        feature  coefficient
35           model_used_unknown    -1.377241
29          provider_used_other    -1.170010
31  model_used_gemini-2.5-flash    -1.086646
8                    char_count     0.975373
30           provider_used_None     0.895954
17           content_type_other    -0.811707
36              model_used_None     0.810748
9                    word_count    -0.702868
19        competition_level_LOW    -0.596158
20     competition_level_MEDIUM    -0.517731


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.